In [130]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

### Load the Data

In [131]:
#relative path
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
data_path = os.path.join(parent_dir, 'Data', 'train.csv')
test_path = os.path.join(parent_dir, 'Data', 'test.csv')

# Load data and test data
data = pd.read_csv(data_path, index_col = 0)
test = pd.read_csv(test_path, index_col = 0)

# Separate data into X and Y
y = data.SalePrice
X = data.drop("SalePrice", axis = 1)

### Examine All Features

In [132]:
print("The shape of X is", X.shape)
print("The shape of test set is", test.shape)
print("The columns of X are:\n", X.columns)

The shape of X is (1460, 79)
The shape of test set is (1459, 79)
The columns of X are:
 Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea',
       'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC',
       'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
       'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt',
       'GarageF

In [133]:
num_name = X.select_dtypes(include=[np.number]).columns
cat_name = X.select_dtypes(include=[np.object_]).columns
print(num_name.shape[0], "numerical variables")
print(cat_name.shape[0], "categorical variables")

36 numerical variables
43 categorical variables


### Missing Value

In [134]:
# Number of empty entries in each column
# Data set
col_missing = X.isnull().sum(axis = 0)
col_missing = col_missing[col_missing > 0]
print(col_missing)
print()

LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64



In [135]:
# Test set
col_missing_test = test.isnull().sum(axis = 0)
col_missing_test = col_missing_test[col_missing_test > 0]
print(col_missing_test)
len(col_missing_test)

MSZoning           4
LotFrontage      227
Alley           1352
Utilities          2
Exterior1st        1
Exterior2nd        1
MasVnrType       894
MasVnrArea        15
BsmtQual          44
BsmtCond          45
BsmtExposure      44
BsmtFinType1      42
BsmtFinSF1         1
BsmtFinType2      42
BsmtFinSF2         1
BsmtUnfSF          1
TotalBsmtSF        1
BsmtFullBath       2
BsmtHalfBath       2
KitchenQual        1
Functional         2
FireplaceQu      730
GarageType        76
GarageYrBlt       78
GarageFinish      78
GarageCars         1
GarageArea         1
GarageQual        78
GarageCond        78
PoolQC          1456
Fence           1169
MiscFeature     1408
SaleType           1
dtype: int64


33

In [136]:
# All rows have missing entry in data set
row_with_missing = [row for index, row in X.iterrows() if row.isnull().any()]
len(row_with_missing)

1460

### Three Feature Sets

#### Set 1: Numerical Variables without missing entries
* Standard LR (mse: 22620)

In [137]:
num_no_missing = num_name.difference(col_missing.index).difference(col_missing_test.index)
# clean features - no missing X in both dataset and test set
clean_feat = list(num_no_missing) + [y.name]
clean_feat_test = list(num_no_missing)
len(clean_feat)

26

In [138]:
num_no_missing

Index(['1stFlrSF', '2ndFlrSF', '3SsnPorch', 'BedroomAbvGr', 'EnclosedPorch',
       'Fireplaces', 'FullBath', 'GrLivArea', 'HalfBath', 'KitchenAbvGr',
       'LotArea', 'LowQualFinSF', 'MSSubClass', 'MiscVal', 'MoSold',
       'OpenPorchSF', 'OverallCond', 'OverallQual', 'PoolArea', 'ScreenPorch',
       'TotRmsAbvGrd', 'WoodDeckSF', 'YearBuilt', 'YearRemodAdd', 'YrSold'],
      dtype='object')

#### Set 2: All Numerical Variables
* Impute Mean (mse: 21098)
* Impute 0 (mse: 21037)
* Impute 0 + log transformation (mse: 18236)

In [139]:
len(num_name)

# clean features
clean_feat = list(num_name) + [y.name]
clean_feat_test = list(num_name)
len(clean_feat)

37

In [140]:
# Missing entry in numerical variables
missing = data[num_name].isnull().sum(axis = 0)
print(missing[missing > 0])
print()

missing = test[num_name].isnull().sum(axis = 0)
print(missing[missing > 0])

LotFrontage    259
MasVnrArea       8
GarageYrBlt     81
dtype: int64

LotFrontage     227
MasVnrArea       15
BsmtFinSF1        1
BsmtFinSF2        1
BsmtUnfSF         1
TotalBsmtSF       1
BsmtFullBath      2
BsmtHalfBath      2
GarageYrBlt      78
GarageCars        1
GarageArea        1
dtype: int64


#### Set 3: Numerical and Categorical Variables without missing entries
* One hot encoding (mae: 17034)
* add ordinal encoding (mae: 16863)

In [141]:
# col_no_missing in both data and test set
col_no_missing = X.columns.difference(col_missing.index).difference(col_missing_test.index)
len(col_no_missing)

45

In [142]:
# Numerical variables are good to go
# Try One Hot Encoding and Ordinal Encoding For Categoricals
cat_no_missing = col_no_missing.difference(num_name)
print(len(cat_no_missing))

20


In [143]:
# One Hot Encoding
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop="first", sparse_output=False) # Handle unknown by default is error

# One Hot Encoding Variables (14 out of 20)
ohe_var = ["BldgType", "CentralAir", "Condition1", "Condition2", "Foundation", 
                                             "Heating", "HouseStyle", "LandContour", "LotConfig", "Neighborhood", 
                                             "RoofMatl", "RoofStyle", "SaleCondition", "Street"]
ord_var = ["ExterCond", "ExterQual", "HeatingQC", "LandSlope", "LotShape", "PavedDrive"] 


# One Hot Encoding Matrix
ohe_matrix = pd.DataFrame(ohe.fit_transform(X[ohe_var]))
ohe_matrix_test = pd.DataFrame(ohe.transform(test[ohe_var])) # fit in the training data so that test data has the same dimension

# Put index and column name back
ohe_matrix.index = X.index
ohe_matrix_test.index = test.index

ohe_columns = ohe.get_feature_names_out(ohe_var).astype(str)
ohe_matrix.columns = ohe_columns
ohe_matrix_test.columns = ohe_columns

# Concat back to "data" and test
data = pd.concat([data, ohe_matrix], axis = 1)
test = pd.concat([test, ohe_matrix_test], axis = 1)

# clean_features
clean_feat = list(col_no_missing.difference(ohe_var).difference(ord_var)) + list(ohe_columns) + [y.name]
clean_feat_test = list(col_no_missing.difference(ohe_var).difference(ord_var)) + list(ohe_columns)
len(clean_feat)

112

In [144]:
# Ordinal Encoding
from sklearn.preprocessing import OrdinalEncoder

# Ordinal Encoding Variables (6 out of 20)
categories = [['Po', 'Fa', 'TA', 'Gd', 'Ex'], 
              ['Po', 'Fa', 'TA', 'Gd', 'Ex'], 
              ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
              ["Sev", "Mod", "Gtl"],
              ["IR3", "IR2", "IR1", "Reg"],
              ["N", "P", "Y"]]
ord_encoder = OrdinalEncoder(categories=categories)

# One Hot Encoding Matrix
ord_matrix = pd.DataFrame(ord_encoder.fit_transform(X[ord_var]))
ord_matrix_test = pd.DataFrame(ord_encoder.transform(test[ord_var]))

# Put index and column name back
ord_matrix.index = X.index
ord_matrix_test.index = test.index

ord_matrix.columns = ord_var
ord_matrix_test.columns = ord_var

# replace "data" and test with ordinal features
data[ord_var] = ord_matrix
test[ord_var] = ord_matrix_test

# One Hot Encoding
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop="first", sparse_output=False) # Handle unknown by default is error

# One Hot Encoding Variables (14 out of 20)
ohe_var = ["BldgType", "CentralAir", "Condition1", "Condition2", "Foundation", 
                                             "Heating", "HouseStyle", "LandContour", "LotConfig", "Neighborhood", 
                                             "RoofMatl", "RoofStyle", "SaleCondition", "Street"]

# One Hot Encoding Matrix
ohe_matrix = pd.DataFrame(ohe.fit_transform(X[ohe_var]))
ohe_matrix_test = pd.DataFrame(ohe.transform(test[ohe_var]))

# Put index and column name back
ohe_matrix.index = X.index
ohe_matrix_test.index = test.index

ohe_columns = ohe.get_feature_names_out(ohe_var).astype(str)
ohe_matrix.columns = ohe_columns
ohe_matrix_test.columns = ohe_columns

# Concat back to "data" and test
data = pd.concat([data, ohe_matrix], axis = 1)
test = pd.concat([test, ohe_matrix_test], axis = 1)

# clean features
clean_feat = list(col_no_missing.difference(ohe_var)) + list(ohe_columns) + [y.name]
clean_feat_test = list(col_no_missing.difference(ohe_var)) + list(ohe_columns)
len(clean_feat)

118

In [166]:
list(cat_no_missing.difference(ohe_var)) #Gives ordinal encoded non missing categorical

['ExterCond', 'ExterQual', 'HeatingQC', 'LandSlope', 'LotShape', 'PavedDrive']

#### Set 4: All numerical, and categorical without missing in both dataset and test set

In [145]:
# clean features
clean_feat = list(num_name) + list(cat_no_missing.difference(ohe_var)) + list(ohe_columns) + [y.name]
clean_feat_test = list(num_name) + list(cat_no_missing.difference(ohe_var)) + list(ohe_columns)
len(clean_feat) # 117 + 11 + 1

129

#### Set 5: All variables

In [19]:
# missing in either data or test set or both
cat_missing = cat_name.difference(cat_no_missing)
print(cat_missing.shape)
data[cat_missing].isnull().sum(axis = 0)
# 0 missing in dataset only has less than 10 missing in test

(23,)


Alley           1369
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
BsmtQual          37
Electrical         1
Exterior1st        0
Exterior2nd        0
Fence           1179
FireplaceQu      690
Functional         0
GarageCond        81
GarageFinish      81
GarageQual        81
GarageType        81
KitchenQual        0
MSZoning           0
MasVnrType       872
MiscFeature     1406
PoolQC          1453
SaleType           0
Utilities          0
dtype: int64

In [162]:
def mean_median(feat:str, args, index):
    '''
    input
        feat: feature name
        args: feat value name in the dataset
        index: row index name
    '''
    assert len(index) == len(args), "length of index and unique feat value is different"

    df = pd.DataFrame({"Mean": [0]*len(index), "Median": [0]*len(args), "n": [0]*len(args)}, index = index)
    for ind, arg in enumerate(args):
        if type(arg) is not list:
            df.iloc[ind, 0] = data[data[feat] == arg].SalePrice.mean()
            df.iloc[ind, 1] = data[data[feat] == arg].SalePrice.median()
            df.iloc[ind, 2] = len(data[data[feat] == arg])
        else:
            df.iloc[ind, 0] = data[data[feat].isin(arg)].SalePrice.mean()
            df.iloc[ind, 1] = data[data[feat].isin(arg)].SalePrice.median()
            df.iloc[ind, 2] = len(data[data[feat].isin(arg)])
    display(df.round(2).sort_values(by="Mean", ascending=False))
    return data[feat].unique()

In [164]:
# Temporal for test set
mean_median("Utilities", list(data.Utilities.unique()),list(data.Utilities.unique()))

,Mean,Median,n
AllPub,180950.96,163000,1459
NoSeWa,137500.00,137500,1


array(['AllPub', 'NoSeWa'], dtype=object)

In [129]:
# Alley
data.Alley.fillna("NoAlley", inplace = True)
mean_median("Alley", ["NoAlley", "Grvl", "Pave"], ["No Alley", "Gravel", "Paved"])

,Mean,Median,n
No Alley,183452.13,165000,1369
Gravel,122219.08,119500,50
Paved,168000.59,172500,41


array(['NoAlley', 'Grvl', 'Pave'], dtype=object)

In [147]:
# BsmtCond
data.BsmtCond.fillna("NA", inplace=True)
mean_median("BsmtCond", ['Gd', 'TA', 'Fa', 'Po', 'NA'], ['Good', 'Typical', 'Fair', 'Poor', "No Basement"])

,Mean,Median,n
Good,213599.91,193879,65
Typical,183632.62,165000,1311
Fair,121809.53,118500,45
Poor,64000.00,64000,2
No Basement,105652.89,101800,37


array(['TA', 'Gd', 'NA', 'Fa', 'Po'], dtype=object)

In [149]:
# BsmtExposure
data.BsmtExposure.fillna("NA", inplace = True)
mean_median("BsmtExposure", ["NA", "No", 'Mn', 'Av', 'Gd'], ["No Basement", "No exposure", "Minimum", 'Average', 'Good'])

,Mean,Median,n
No Basement,107938.34,104025,38
No exposure,165652.30,154000,953
Minimum,192789.66,182450,114
Average,206643.42,185850,221
Good,257689.81,226975,134


array(['No', 'Gd', 'Mn', 'Av', 'NA'], dtype=object)

In [26]:
# one exception of no basement but with a typical condition?
data[((data.BsmtExposure=="NA") & (data.BsmtCond!="NA"))][["BsmtExposure", "BsmtCond"]]

,BsmtExposure,BsmtCond
Id,,
949,NA,TA


In [169]:
# BsmtFinType1
data.BsmtFinType1.fillna("NA", inplace=True)
mean_median("BsmtFinType1", ['GLQ', 'ALQ', 'BLQ', 'Rec', 'LwQ', 'Unf', "NA"], ["good", 'average quarter', 'below average', 'average rec', 'low', 'unfinished','no basement'])
# Same as BsmtCond in this separation

,Mean,Median,n
good,235413.72,213750,418
unfinished,170670.58,161750,430
average quarter,161573.07,149250,220
low,151852.70,139000,74
below average,149493.66,139100,148
average rec,146889.25,142000,133
no basement,105652.89,101800,37


array(['GLQ', 'ALQ', 'Unf', 'Rec', 'BLQ', 'NA', 'LwQ'], dtype=object)

In [76]:
# BsmtFinType2
data.BsmtFinType2.fillna("NA", inplace=True)
mean_median("BsmtFinType2", ["NA", ['GLQ', 'ALQ', 'Unf', 'Rec', 'BLQ', 'LwQ']], ["No Basement", "Has finished area"])

,Mean,Median,n
No Basement,110346.24,104025,38
Has finished area,182807.17,165000,1422


array(['Unf', 'BLQ', 'NA', 'ALQ', 'Rec', 'LwQ', 'GLQ'], dtype=object)

In [150]:
# BsmtQual
data.BsmtQual.fillna("NA", inplace=True)
mean_median("BsmtQual", ["Ex", 'Gd', 'TA', 'Fa','Po', 'NA'], ['Excellent', 'Good', 'Typical', 'Fair','Poor', "No Basement"])


,Mean,Median,n
Excellent,327041.04,318000.0,121
Good,202688.48,192070.0,618
Typical,140759.82,135500.0,649
Fair,115692.03,112000.0,35
Poor,NaN,NaN,0
No Basement,105652.89,101800.0,37


array(['Gd', 'TA', 'Ex', 'NA', 'Fa'], dtype=object)

In [88]:
# Electrical
print(len(data[data.Electrical.isnull()]), "missing in Electrical columns")
print(data.Electrical.value_counts())
data.Electrical.fillna(data.Electrical.mode()[0], inplace=True)

1 missing in Electrical columns
Electrical
SBrkr    1334
FuseA      94
FuseF      27
FuseP       3
Mix         1
Name: count, dtype: int64


In [151]:
# Fence 
data.Fence.fillna("NA", inplace=True)
mean_median("Fence", ["GdPrv", "MnPrv", "GdWo","MnWw", "NA"], ['Good Priv', 'Min Priv', 'Good Wood', 'MinWood', "No Fence"])

,Mean,Median,n
Good Priv,178927.46,167500,59
Min Priv,148751.09,137450,157
Good Wood,140379.31,138750,54
MinWood,134286.36,130000,11
No Fence,187596.84,173000,1179


array(['NA', 'MnPrv', 'GdWo', 'GdPrv', 'MnWw'], dtype=object)

In [152]:
# FireplaceQu
data.FireplaceQu.fillna("NA", inplace=True)
mean_median("FireplaceQu", ["Ex", 'Gd', 'TA', 'Fa', 'Po', 'NA'], ['Excellent', 'Good', 'Typical', 'Fair', 'Poor', "No FirePlace"])

,Mean,Median,n
Excellent,337712.50,314250,24
Good,226351.42,206950,380
Typical,205723.49,187500,313
Fair,167298.48,158000,33
Poor,129764.15,131500,20
No FirePlace,141331.48,135000,690


array(['NA', 'TA', 'Gd', 'Fa', 'Ex', 'Po'], dtype=object)

In [94]:
# GarageFinish
data.GarageFinish.fillna("NA", inplace=True)
mean_median("GarageFinish", ["NA", "Unf", "RFn", "Fin"], ["No Garage", "Unfinished", "PartialFin", "Finished"])

,Mean,Median,n
No Garage,103317.28,100000,81
Unfinished,142156.42,135000,605
PartialFin,202068.87,190000,422
Finished,240052.69,215000,352


array(['RFn', 'Unf', 'Fin', 'NA'], dtype=object)

In [97]:
# GarageCond
data.GarageCond.fillna("NA", inplace=True)
mean_median("GarageCond", ["Ex", 'Gd', 'TA', 'Fa', 'Po', 'NA'], ['Excellent', 'Good', 'Typical', 'Fair', 'Poor', 'No Garage'])

,Mean,Median,n
Excellent,124000.00,124000,2
Good,179930.00,148000,9
Typical,187885.74,170000,1326
Fair,114654.03,114504,35
Poor,108500.00,108000,7
No Garage,103317.28,100000,81


array(['TA', 'Fa', 'NA', 'Gd', 'Po', 'Ex'], dtype=object)

In [96]:
# GarageQual
data.GarageQual.fillna("NA", inplace=True)
mean_median("GarageQual", ["Ex", 'Gd', 'TA', 'Fa', 'Po', 'NA'], ['Excellent', 'Good', 'Typical', 'Fair', 'Poor', 'No Garage'])

,Mean,Median,n
Excellent,241000.00,127500,3
Good,215860.71,209115,14
Typical,187489.84,170000,1311
Fair,123573.35,115000,48
Poor,100166.67,96500,3
No Garage,103317.28,100000,81


array(['TA', 'Fa', 'Gd', 'NA', 'Ex', 'Po'], dtype=object)

In [98]:
# GarageType
data.GarageType.fillna("NA", inplace=True)
mean_median("GarageType", ['2Types', 'Attchd', 'Basment', 'BuiltIn', 'CarPort', 'Detchd', 'NA'], [">1", 'attached', 'basement', 'builtin', 'CarPort', 'Detached', 'No Garage'])

,Mean,Median,n
>1,151283.33,159000,6
attached,202892.66,185000,870
basement,160570.68,148000,19
builtin,254751.74,227500,88
CarPort,109962.11,108000,9
Detached,134091.16,129500,387
No Garage,103317.28,100000,81


array(['Attchd', 'Detchd', 'BuiltIn', 'CarPort', 'NA', 'Basment',
       '2Types'], dtype=object)

In [154]:
# MasVnrType
data.MasVnrType.fillna('NA', inplace = True)
has_masonry_veneer = list(data.MasVnrType.unique())
has_masonry_veneer.remove("NA")
mean_median("MasVnrType", ["NA", 'Stone', 'BrkFace', 'BrkCmn'], ["No Masonry Veneer", "Stone", 'BrickFace', 'BrickCommon'])

,Mean,Median,n
No Masonry Veneer,156958.24,143125,872
Stone,265583.62,246839,128
BrickFace,204691.87,181000,445
BrickCommon,146318.07,139000,15


array(['BrkFace', 'NA', 'Stone', 'BrkCmn'], dtype=object)

In [122]:
# MiscFeature
data.MiscFeature.fillna('NA', inplace = True)
mean_median("MiscFeature", ["NA", "Tenc", "Shed", "Othr", "Gar2", "Elev"], ["None", 'Tennis', 'Shed', 'Other', 'Gar2', 'Elevator'])

,Mean,Median,n
None,182046.41,164250.0,1406
Tennis,NaN,NaN,0
Shed,151187.61,144000.0,49
Other,94000.00,94000.0,2
Gar2,170750.00,170750.0,2
Elevator,NaN,NaN,0


array(['NA', 'Shed', 'Gar2', 'Othr', 'TenC'], dtype=object)

In [123]:
# PoolQC
data.PoolQC.fillna('NA', inplace = True)
mean_median("PoolQC", ["Ex", 'Gd', 'TA', 'Fa', 'NA'], ['Excellent', 'Good', 'Typical', 'Fair', 'No Pool'])

,Mean,Median,n
Excellent,490000.00,490000.0,2
Good,201990.00,171000.0,3
Typical,NaN,NaN,0
Fair,215500.00,215500.0,2
No Pool,180404.66,162900.0,1453


array(['NA', 'Ex', 'Fa', 'Gd'], dtype=object)

In [127]:
# if even number, take the average of the middle two as the median
data[data.PoolQC == "Ex"].SalePrice

Id
198     235000
1183    745000
Name: SalePrice, dtype: int64

In [ ]:
categories = [["NA", 'Po', 'Fa', 'TA', 'Gd', 'Ex'], # BsmtCond
              ['NA', 'No', 'Mn', 'AV', 'Gd'], # BsmtExposure
              ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', "GLQ"], # BsmtFinType1
              ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', "GLQ"], # BsmtFinType2
              ["NA", 'Po', 'Fa', 'TA', 'Gd', 'Ex'], # BsmtQual

              ["Sev", "Mod", "Gtl"], 
              ["IR3", "IR2", "IR1", "Reg"],
              ["N", "P", "Y"]]
ord_encoder_miss = OrdinalEncoder(categories=categories)

In [27]:
test[cat_missing].isnull().sum(axis = 0)

Alley           1352
BsmtCond          45
BsmtExposure      44
BsmtFinType1      42
BsmtFinType2      42
BsmtQual          44
Electrical         0
Exterior1st        1
Exterior2nd        1
Fence           1169
FireplaceQu      730
Functional         2
GarageCond        78
GarageFinish      78
GarageQual        78
GarageType        76
KitchenQual        1
MSZoning           4
MasVnrType       894
MiscFeature     1408
PoolQC          1456
SaleType           1
Utilities          2
dtype: int64

In [168]:
mean_median("KitchenQual", data.KitchenQual.unique(), data.KitchenQual.unique())

,Mean,Median,n
Ex,328554.67,316750,100
Gd,212116.02,201400,586
TA,139962.51,137000,735
Fa,105565.21,115000,39


array(['Gd', 'TA', 'Ex', 'Fa'], dtype=object)

In [167]:
data.KitchenQual.unique()

array(['Gd', 'TA', 'Ex', 'Fa'], dtype=object)

### Save Clean Data

In [135]:
clean_data_path = os.path.join(parent_dir, 'Data', 'train_clean.csv')
clean_test_path = os.path.join(parent_dir, 'Data', 'test_clean.csv')

data[clean_feat].to_csv(clean_data_path)
test[clean_feat_test].to_csv(clean_test_path)